In [1]:
import joblib
import pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, OrdinalEncoder, FunctionTransformer
from xgboost import XGBRegressor

# ✅ Import your custom components
from custom_transformers import binary_map, FrequencyEncoder

# 1. Load data
df = pd.read_csv("preprocessed/cars_preprocessed.csv")
y = df["Price"]
X = df.drop(columns=["Price"])

# 2. Define column types
binary_cols = ['Première_main', 'Transmission']
ordinal_col = ['État']
ordinal_order = [['mauvais', 'correct', 'bon', 'très bon', 'excellent', 'neuf']]
onehot_cols = ['Carburant', 'Origine']
freq_cols = ['Marque', 'Model']
passthrough_cols = ['Nb_portes']
numerical_cols = ['Kilométrage', 'Puissance']

# Optional: create Car_Age if needed
X["Car_Age"] = 2025 - X["Model_Year"]
numerical_cols.append("Car_Age")
X.drop(columns=["Model_Year"], inplace=True)

# 3. Column Transformer
preprocessor = ColumnTransformer(transformers=[
    ('num', StandardScaler(), numerical_cols),
    ('binary', FunctionTransformer(binary_map), binary_cols),
    ('ordinal', OrdinalEncoder(categories=ordinal_order), ordinal_col),
    ('onehot', OneHotEncoder(handle_unknown='ignore'), onehot_cols),
    ('freq', FrequencyEncoder(), freq_cols),
    ('pass', 'passthrough', passthrough_cols)
])

# 4. Final pipeline
pipeline = Pipeline(steps=[
    ('prep', preprocessor),
    ('model', XGBRegressor(
        n_estimators=400,
        learning_rate=0.05,
        max_depth=6,
        subsample=0.8,
        colsample_bytree=0.8,
        reg_alpha=0.5,
        reg_lambda=2,
        random_state=42
    ))
])

# 5. Fit the model
pipeline.fit(X, y)

# 6. Save pipeline
joblib.dump(pipeline, "best_xgb_model.joblib", compress=3)
print("✅ Pipeline saved successfully!")


C:\Users\anass\Documents\Licence Ex\S6\machine learning\Projet\LAST TRY\ANASS\car_price_prediction\models\custom_transformers.py:6: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  return X.replace({


✅ Pipeline saved successfully!
